In [1]:
## Script for processing of PsychoPy data and output files in BIDS format
# Run with python 3.10 fMRI env
# James Fodor 2023

import numpy as np
import csv
import pandas as pd
import json

# set display options
np.set_printoptions(precision=5, threshold=5000, linewidth=220, suppress=True, floatmode='fixed')
pd.set_option('display.max_rows', 500)

C:\Users\fods1\AppData\Local\Temp\ipykernel_24268\3668994657.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### Load results from psychopy output file

In [4]:
# Load psychopy log file into numpy array
subject_id = '44'
rows = []
with open('../Data-behavioural/Psychopy/sub-'+subject_id+'/all_runs.csv', 'r') as file:
    for row in csv.reader(file):
        rows.append([cell.replace('\n', ' ') for cell in row])
psychopy_data_np = np.array(rows)
print(psychopy_data_np.shape)

# Define params
num_runs = 8
TR = 0.8
num_TR = 636

(435, 72)


In [5]:
# Save as pandas dataframe
start_index = 0 # in case we need to exclude some columns
df = pd.DataFrame(psychopy_data_np[2:,start_index:], columns=psychopy_data_np[0,start_index:])

# Remove rows without run index
df["runs.thisIndex"] = df["runs.thisIndex"].replace("", np.nan)
df = df.dropna(subset=["runs.thisIndex"]).copy()

# Get number of sentences
sentence_ids = df['sentence_id'].values
sentence_ids = sentence_ids[sentence_ids.astype(bool)]
sentence_ids = sentence_ids.astype(int)
num_sentences = np.max(sentence_ids)

# Show all available data
print(df.columns)

Index(['﻿run_file', 'intro_text', 'final_fixation_time', 'sentence_id',
       'sentence_text', 'sentence_duration', 'interstim_duration',
       'question_duration', 'question_text', 'answer_index', 'option_1',
       'option_2', 'option_3', 'option_4', 'runs.thisRepN', 'runs.thisTrialN',
       'runs.thisN', 'runs.thisIndex', 'trials.thisRepN', 'trials.thisTrialN',
       'trials.thisN', 'trials.thisIndex', 'welcome_text.started',
       'mouse.started', 'mouse.x', 'mouse.y', 'mouse.leftButton',
       'mouse.midButton', 'mouse.rightButton', 'mouse.time',
       'ready_key_resp.started', 'next_run_text.started',
       'ready_key_resp.keys', 'key_resp.started',
       'fixation_text_trigger.started', 'key_resp.keys', 'key_resp.rt',
       'key_resp.duration', 'sentence.started', 'sentence.stopped',
       'question_fulltext.started', 'question_option_1.started',
       'question_option_2.started', 'question_option_3.started',
       'question_option_4.started', 'question_response.sta

In [6]:
# Get start time for each run so that times can be adjusted to be relative to each run 'trigger' time
run_start_times = df[['fixation_text_trigger.started','key_resp.rt']].values
run_start_times = [float(pair[0])+float(pair[1]) for pair in run_start_times if pair[0] != '']
run_start_times = {i:run_start_times[i] for i in range(len(run_start_times))}
print('Run start times:',run_start_times)

# Convert relevant columns to numeric types
df["runs.thisIndex"] = df["runs.thisIndex"].astype(int)
df["sentence.started"] = df["sentence.started"].astype(float)
df["sentence_duration"] = df["sentence_duration"].astype(float)
df["question_fulltext.started"] = df["question_fulltext.started"].astype(float)

# Adjust times to be relative to run start times
df["sentence.started"] = df["sentence.started"] - df["runs.thisIndex"].map(run_start_times)
df["question_fulltext.started"] = df["question_fulltext.started"] - df["runs.thisIndex"].map(run_start_times)

Run start times: {0: 32.693438999995124, 1: 577.5329262000159, 2: 1120.5391711000702, 3: 1666.200002800033, 4: 2213.3017072000075, 5: 2756.5988292000256, 6: 3301.094451699988, 7: 3852.092805000022}


### Print results in array for visualisation

In [7]:
# Prepare list to accumulate new rows
new_rows = []

# Loop through each row and create separate rows for sentences and questions
for _, row in df.iterrows():
    # --- Sentence row ---
    new_rows.append({
        # Common metadata
        "runs.thisIndex": row["runs.thisIndex"],
        "onset": row["sentence.started"],
        "duration": row["sentence_duration"],
        "stim_type": "sentence",
        "sentence_id": row.get("sentence_id", np.nan),
        "stimulus_text": row["sentence_text"],
        # Sentence metadata
        "response_time": np.nan,
        "answer_index": np.nan,
        "question_response": np.nan
    })
    
    # --- Question row (only if applicable) ---
    if row.get("answer_index") != "None":
        new_rows.append({
            # Common metadata
            "runs.thisIndex": row["runs.thisIndex"],
            "onset": row["question_fulltext.started"],
            "duration": row["question_duration"],
            "stim_type": "question",
            "sentence_id": np.nan,
            # Question metadata
            "stimulus_text": row["question_text"],
            "response_time": row.get("question_response.rt", np.nan),
            "answer_index": row.get("answer_index", np.nan),
            "question_response": row.get("question_response.keys", np.nan)
        })

# Convert list of dicts back into a DataFrame
df_split = pd.DataFrame(new_rows)

# Convert numeric columns to appropriate types
df_split["onset"] = pd.to_numeric(df_split["onset"], errors="coerce")
df_split["duration"] = pd.to_numeric(df_split["duration"], errors="coerce")
df_split["response_time"] = pd.to_numeric(df_split["response_time"], errors="coerce")

# Optional: sort by run and onset
df_split = df_split.sort_values(["runs.thisIndex", "onset"]).reset_index(drop=True)

In [8]:
df_split

,runs.thisIndex,onset,duration,stim_type,sentence_id,stimulus_text,response_time,answer_index,question_response
0,0,0.042945,3.016967,sentence,58,The dog noticed the strange object near the ch...,NaN,NaN,NaN
1,0,3.142199,5.000000,question,NaN,Who saw the strange object?,1.801928,C,c
2,0,13.641324,3.134807,sentence,66,The father objected to the unpleasant smell of...,NaN,NaN,NaN
3,0,23.223476,3.490072,sentence,39,The government purchased the apartment block f...,NaN,NaN,NaN
4,0,29.946379,3.252939,sentence,64,The children objected to the unpleasant smell ...,NaN,NaN,NaN
5,0,37.187134,3.668685,sentence,46,The family blocked the purchase of the apartme...,NaN,NaN,NaN
6,0,40.861650,5.000000,question,NaN,What was the company trying to buy?,2.866938,B,b
7,0,48.335972,3.312113,sentence,19,The psychologist spoke to the secretary about ...,NaN,NaN,NaN
8,0,54.051014,4.510849,sentence,69,The children rarely objected to the unpleasant...,NaN,NaN,NaN
9,0,63.118406,5.428959,sentence,78,In response to recent events the lobbyist quic...,NaN,NaN,NaN


### Extract sentence and question onset and offset times

In [185]:
# Iterate over runs
for run_id, run_df in df_split.groupby("runs.thisIndex"):
    # Drop the run column (which is now redundant)
    run_df_out = run_df.drop(columns=["runs.thisIndex"])

    # Create a BIDS-style filename (zero-padded)
    output_file = f"sub-"+subject_id+"_task-sentences_run-"+str(int(run_id)+1)+"_events.tsv"
    
    # Save as TSV
    run_df_out.to_csv(output_file, sep="\t", index=False, float_format="%08.4f", na_rep="nan")
    
    print(f"Saved run {run_id} to {output_file}")


Saved run 0 to sub-44_task-sentences_run-1_events.tsv
Saved run 1 to sub-44_task-sentences_run-2_events.tsv
Saved run 2 to sub-44_task-sentences_run-3_events.tsv
Saved run 3 to sub-44_task-sentences_run-4_events.tsv
Saved run 4 to sub-44_task-sentences_run-5_events.tsv
Saved run 5 to sub-44_task-sentences_run-6_events.tsv
Saved run 6 to sub-44_task-sentences_run-7_events.tsv
Saved run 7 to sub-44_task-sentences_run-8_events.tsv
